# PANDORA preprocessing

Run the cells in order. Change the paths below if needed. The raw file is read in chunks, but the selected comments are kept in memory.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import KFold

DATA = Path("../data")  # From the project's src folder
PROFILES = DATA / "author_profiles.csv"
COMMENTS = DATA / "all_comments_since_2015.csv"
OUTPUT = DATA / "preprocessed"

MIN_COMMENTS = 20
ENGLISH_ONLY = True  # Set to False to keep all languages
SEED = 42

BIG5 = ["agreeableness", "openness", "conscientiousness",
        "extraversion", "neuroticism"]

PERSONALITY_SUBREDDITS = {
    "mbti", "mbtiplus", "mbtimemes", "mbtitypeme", "myersbriggs",
    "16personalities", "personalitycafe", "personalitytypes",
    "personalitytest", "personalitytests", "bigfive", "big5",
    "enneagram", "enneagrammemes", "socionics", "cognitivefunctions",
    "intp", "infj", "infp", "intj", "entp", "enfp", "enfj", "entj",
    "istj", "isfj", "istp", "isfp", "estj", "esfj", "estp", "esfp",
}

quality = {}

## 1. Check the Big Five scores

Look at the score ranges and test labels. The same range can mean different things on different tests. Keep the original scores and check the audit before pooling them.

In [2]:
profiles = pd.read_csv(PROFILES)
quality["profile_rows"] = len(profiles)
profiles["author"] = profiles["author"].str.strip().str.lower()

profiles[BIG5] = profiles[BIG5].apply(pd.to_numeric, errors="coerce")
profiles[BIG5] = profiles[BIG5].replace([float("inf"), float("-inf")], pd.NA)
scale_summary = profiles[BIG5].describe().T
scale_summary["missing_or_invalid"] = profiles[BIG5].isna().sum()
display(scale_summary)

scale_fields = ["type", "is_description", "is_percentile", "is_score",
                "test_name", "predicted_test", "test_scale", "test_result_type"]
scale_fields = [column for column in scale_fields if column in profiles]
scale_groups = []
for column in scale_fields:
    summary = profiles.groupby(column, dropna=False)[BIG5].agg(
        ["count", "min", "median", "max"]
    )
    summary.columns = [f"{trait}_{stat}" for trait, stat in summary.columns]
    summary = summary.reset_index().rename(columns={column: "value"})
    summary.insert(0, "field", column)
    scale_groups.append(summary)
    display(summary.head())

,count,mean,std,min,25%,50%,75%,max,missing_or_invalid
agreeableness,1606.0,42.396949,31.041252,0.0,14.00,40.0,70.000,100.0,8689
openness,1588.0,62.454345,27.780811,0.0,43.75,69.0,85.000,100.0,8707
conscientiousness,1605.0,40.162617,30.394938,0.0,13.00,35.0,65.000,99.0,8690
extraversion,1608.0,37.375000,30.478740,0.0,10.00,30.0,61.125,99.0,8687
neuroticism,1603.0,49.783219,32.374254,0.0,18.50,50.0,82.000,100.0,8692


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,type,description,149,1.0,50.0,96.0,150,3.0,74.0,95.0,...,35.0,95.0,150,2.0,38.0,97.0,148,1.0,50.0,97.0
1,type,percentiles,635,0.0,39.0,100.0,635,0.0,75.0,100.0,...,35.0,99.0,635,0.0,28.0,99.0,635,0.0,56.0,99.0
2,type,scores,822,0.0,38.0,99.0,803,0.0,61.0,99.0,...,35.0,99.0,823,0.0,30.0,98.0,820,0.0,44.0,100.0
3,type,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,is_description,1.0,150,1.0,50.0,96.0,151,3.0,74.0,95.0,...,35.0,95.0,151,2.0,46.0,97.0,149,1.0,50.0,97.0
1,is_description,NaN,1456,0.0,38.0,100.0,1437,0.0,69.0,100.0,...,35.0,99.0,1457,0.0,30.0,99.0,1454,0.0,48.0,100.0


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,is_percentile,1,367,0.0,32.0,100.0,367,0.0,78.0,99.0,...,31.0,99.0,367,0.0,31.0,99.0,367,0.0,55.0,99.0
1,is_percentile,q,1,50.0,50.0,50.0,1,90.0,90.0,90.0,...,15.0,15.0,1,15.0,15.0,15.0,1,90.0,90.0,90.0
2,is_percentile,NaN,1238,0.0,41.0,100.0,1220,0.0,68.0,100.0,...,35.0,99.0,1240,0.0,30.0,99.0,1235,0.0,49.0,100.0


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,is_score,1.0,1094,0.0,40.0,100.0,1075,0.0,65.0,100.0,...,35.0,99.0,1095,0.0,30.0,99.0,1092,0.0,46.0,100.0
1,is_score,NaN,512,0.0,35.0,100.0,513,0.0,75.0,99.0,...,32.0,99.0,513,0.0,30.0,99.0,511,0.0,53.0,99.0


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,test_name,123test,17,10.0,10.0,80.0,17,30.0,70.0,90.0,...,50.0,80.0,17,10.0,20.00,30.0,17,30.0,60.0,90.0
1,test_name,bfi2,1,62.0,62.0,62.0,1,81.0,81.0,81.0,...,73.0,73.0,1,56.0,56.00,56.0,1,65.0,65.0,65.0
2,test_name,hexaco,50,0.0,36.0,99.0,50,15.0,83.5,99.0,...,52.0,98.0,50,0.0,8.50,89.0,50,0.0,15.0,97.0
3,test_name,ipip120,202,0.0,50.0,100.0,202,1.0,69.0,100.0,...,42.0,98.0,202,0.0,32.75,99.0,201,1.0,58.0,100.0
4,test_name,ipip300,77,0.0,58.0,99.0,77,1.0,73.0,99.0,...,31.0,96.0,77,0.0,17.00,99.0,77,1.0,52.0,99.0


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,predicted_test,0.0,1009,0.0,38.0,100.0,996,0.0,69.0,100.0,...,35.0,99.0,1011,0.0,24.0,99.0,1007,0.0,54.0,99.0
1,predicted_test,1.0,597,0.0,41.0,99.0,592,0.0,68.5,99.0,...,35.0,99.0,597,0.0,43.0,98.0,596,0.0,40.0,100.0
2,predicted_test,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,test_scale,percentiles,1606,0.0,40.0,100.0,1588,0.0,69.0,100.0,...,35.0,99.0,1608,0.0,30.0,99.0,1603,0.0,50.0,100.0
1,test_scale,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN


,field,value,agreeableness_count,agreeableness_min,agreeableness_median,agreeableness_max,openness_count,openness_min,openness_median,openness_max,...,conscientiousness_median,conscientiousness_max,extraversion_count,extraversion_min,extraversion_median,extraversion_max,neuroticism_count,neuroticism_min,neuroticism_median,neuroticism_max
0,test_result_type,description,149,1.0,50.0,96.0,150,3.0,74.0,95.0,...,35.0,95.0,150,2.0,38.0,97.0,148,1.0,50.0,97.0
1,test_result_type,percentiles,758,0.0,43.0,100.0,758,0.0,75.0,100.0,...,38.0,99.0,758,0.0,30.5,99.0,758,0.0,53.0,100.0
2,test_result_type,scores,699,0.0,34.0,99.0,680,0.0,61.0,99.0,...,32.0,99.0,700,0.0,30.0,98.0,697,0.0,43.0,99.0
3,test_result_type,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,...,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN


In [3]:
valid_author = profiles["author"].fillna("").ne("")
valid_author &= ~profiles["author"].isin(["[deleted]", "[removed]"])
complete = profiles.loc[valid_author, ["author"] + BIG5].dropna()
quality["complete_profile_rows"] = len(complete)

# Drop users with conflicting scores.
big5_profiles = complete.drop_duplicates()
conflicting = big5_profiles["author"].duplicated(keep=False)
quality["users_with_conflicting_scores"] = big5_profiles.loc[
    conflicting, "author"
].nunique()
big5_profiles = big5_profiles.loc[~conflicting].sort_values("author")
quality["users_with_complete_scores"] = len(big5_profiles)
display(big5_profiles[BIG5].describe().T)

,count,mean,std,min,25%,50%,75%,max
agreeableness,1568.0,42.147003,30.966034,0.0,13.00,39.25,70.0,100.0
openness,1568.0,62.536033,27.807625,0.0,43.75,69.00,86.0,100.0
conscientiousness,1568.0,40.232143,30.374644,0.0,13.00,35.00,65.0,99.0
extraversion,1568.0,37.147321,30.460500,0.0,10.00,30.00,60.0,99.0
neuroticism,1568.0,49.874043,32.367686,0.0,18.75,50.00,82.0,100.0


## 2. Read and clean the comments

In [4]:
columns = ["id", "author", "subreddit", "body"]
if ENGLISH_ONLY:
    columns.append("lang")

parts = []
quality["comments_read"] = 0
quality["comments_from_big5_users"] = 0
quality["comments_after_language_filter"] = 0
quality["empty_or_removed_comments"] = 0
quality["comments_with_invalid_id_or_subreddit"] = 0

for chunk in pd.read_csv(COMMENTS, usecols=columns, dtype="string",
                         keep_default_na=False, chunksize=200_000):
    quality["comments_read"] += len(chunk)
    chunk["author"] = chunk["author"].str.strip().str.lower()
    chunk = chunk[chunk["author"].isin(big5_profiles["author"])].copy()
    quality["comments_from_big5_users"] += len(chunk)

    if ENGLISH_ONLY:
        chunk = chunk[chunk["lang"].str.strip().str.lower().eq("en")].copy()
    quality["comments_after_language_filter"] += len(chunk)

    chunk["body"] = chunk["body"].str.strip()
    removed = chunk["body"].str.lower().isin(["", "[deleted]", "[removed]"])
    quality["empty_or_removed_comments"] += int(removed.sum())
    chunk = chunk.loc[~removed].copy()

    chunk["id"] = chunk["id"].str.strip().str.lower().str.removeprefix("t1_")
    chunk["subreddit"] = (
        chunk["subreddit"].str.strip().str.lower().str.replace(r"^/?r/", "", regex=True)
    )
    valid = chunk["id"].str.fullmatch("[a-z0-9]+") & chunk["subreddit"].str.fullmatch("[a-z0-9_]+")
    quality["comments_with_invalid_id_or_subreddit"] += int((~valid).sum())
    parts.append(chunk.loc[valid, ["id", "author", "subreddit", "body"]])

comments = pd.concat(parts, ignore_index=True)
del parts
print(f"{len(comments):,} comments before deduplication")

2,830,265 comments before deduplication


## 3. Remove duplicate IDs

Keep the first valid row for each ID. Drop IDs that point to different authors or subreddits. Keep repeated text when the IDs differ.

In [5]:
repeated = comments[comments["id"].duplicated(keep=False)]
id_check = repeated.groupby("id")[["author", "subreddit"]].nunique()
conflict_ids = id_check.index[id_check.gt(1).any(axis=1)]
quality["conflicting_ids"] = len(conflict_ids)

conflicting = comments["id"].isin(conflict_ids)
quality["rows_with_conflicting_ids"] = int(conflicting.sum())
comments = comments.loc[~conflicting]
before_dedup = len(comments)
comments = comments.drop_duplicates("id")
quality["duplicate_rows_removed"] = before_dedup - len(comments)
quality["valid_unique_comments"] = len(comments)
quality["users_with_valid_comments"] = comments["author"].nunique()

## 4. Pick the users

Count comments after removing personality subreddits. Keep users with at least 20 comments. Use those same users in both versions.

In [6]:
personality = comments["subreddit"].isin(PERSONALITY_SUBREDDITS)
removed_by_subreddit = comments.loc[personality, "subreddit"].value_counts()
main = comments.loc[~personality]
activity = main["author"].value_counts()
keep_users = activity.index[activity >= MIN_COMMENTS]

quality["comments_removed_by_subreddit_filter"] = int(personality.sum())
quality["users_after_subreddit_filter"] = len(activity)
quality["users_after_minimum"] = len(keep_users)

main = main[main["author"].isin(keep_users)]
all_subreddits = comments[comments["author"].isin(keep_users)]
users = big5_profiles[big5_profiles["author"].isin(keep_users)]
quality["main_comments_removed_by_minimum"] = int(activity.sum()) - len(main)
display(removed_by_subreddit)

subreddit
mbti          49910
entp          41213
intp          35271
infj          21601
intj          12220
infp           9750
enfp           4854
entj           4409
mbtiplus       1362
enneagram      1337
enfj           1258
istp            923
isfp            673
estp            630
istj            318
esfp            225
estj            220
isfj            195
mbtitypeme      109
socionics       107
esfj             67
mbtimemes        24
Name: count, dtype: Int64

## 5. Make the CV folds

The inner folds use only the outer training users. Fit TF-IDF and any graph model later, inside the training folds. For node2vec, decide how to handle unseen users before evaluation.

In [7]:
if len(users) < 7:
    display(pd.Series(quality))
    raise ValueError("Too few users for nested 5-fold CV.")

outer_folds = users[["author"]].reset_index(drop=True)
inner_tables = []
outer_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(outer_folds)):
    outer_folds.loc[test_idx, "outer_fold"] = fold
    inner = outer_folds.loc[train_idx, ["author"]].reset_index(drop=True)
    inner["outer_fold"] = fold

    inner_cv = KFold(n_splits=5, shuffle=True, random_state=SEED + fold + 1)
    for inner_fold, (_, validation_idx) in enumerate(inner_cv.split(inner)):
        inner.loc[validation_idx, "inner_fold"] = inner_fold
    inner_tables.append(inner.astype({"inner_fold": int}))

outer_folds["outer_fold"] = outer_folds["outer_fold"].astype(int)
inner_folds = pd.concat(inner_tables, ignore_index=True)
display(outer_folds["outer_fold"].value_counts().sort_index())

outer_fold
0    263
1    262
2    262
3    262
4    262
Name: count, dtype: int64

## 6. Save the files

In [8]:
OUTPUT.mkdir(parents=True, exist_ok=False)

for name, table in [("main", main), ("all_subreddits", all_subreddits)]:
    folder = OUTPUT / name
    folder.mkdir()
    counts = table.groupby(["author", "subreddit"]).size().reset_index(name="comment_count")
    user_activity = table["author"].value_counts().rename("comment_count")
    table.to_csv(folder / "comments.csv", index=False)
    users.join(user_activity, on="author").to_csv(folder / "users.csv", index=False)
    counts.to_csv(folder / "subreddit_counts.csv", index=False)
    quality[f"{name}_comments"] = len(table)
    quality[f"{name}_subreddits"] = table["subreddit"].nunique()

outer_folds.to_csv(OUTPUT / "outer_folds.csv", index=False)
inner_folds.to_csv(OUTPUT / "inner_folds.csv", index=False)
scale_summary.to_csv(OUTPUT / "scale_audit.csv", index_label="trait")
if scale_groups:
    pd.concat(scale_groups, ignore_index=True).to_csv(
        OUTPUT / "scale_audit_by_metadata.csv", index=False
    )
removed_by_subreddit.to_csv(OUTPUT / "removed_by_subreddit.csv", header=["comment_count"])
pd.Series(quality, name="count").to_csv(OUTPUT / "quality_report.csv", index_label="step")
display(pd.Series(quality))

profile_rows                                10295
complete_profile_rows                        1568
users_with_conflicting_scores                   0
users_with_complete_scores                   1568
comments_read                            17640062
comments_from_big5_users                  3006566
comments_after_language_filter            2830311
empty_or_removed_comments                      40
comments_with_invalid_id_or_subreddit           6
conflicting_ids                                 0
rows_with_conflicting_ids                       0
duplicate_rows_removed                     304622
valid_unique_comments                     2525643
users_with_valid_comments                    1568
comments_removed_by_subreddit_filter       186676
users_after_subreddit_filter                 1551
users_after_minimum                          1311
main_comments_removed_by_minimum             1696
main_comments                             2337271
main_subreddits                             15309
